# DP2 — DiaObject / DiaSource catalog query over a DDF via Butler

**Author:** dagoret  
**Date:** 2026-03-13  

## Purpose

Retrieve **DiaObject**, **DiaSource**, and **ForcedSourceOnDiaObject** catalogs
for a user-selected Deep Drilling Field (DDF) using **Butler Gen3 only**
(no TAP service, not yet available for DP2 at USDF).

## Actual schema (discovered at runtime — COSMOS, LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545)

- `dia_object` : dims `(skymap, tract)` — 1 ref per tract, index = `diaObjectId`
- `dia_source` : dims `(skymap, tract)` — 1 ref per tract, index = `diaSourceId` (join on `diaObjectId` column)
- `dia_object_forced_source` : dims `(skymap, tract, patch)` — **21 refs per tract** (one per patch)
  - Must iterate over all patch refs and concat
  - Contains per-visit forced photometry linked to DiaObjects

## Reference notebooks
- `2026-03-07_AccessToDP2.ipynb` — Butler setup
- `2026-03-13_DP2_DDF_Tracts_SkyMap_Mollweide.ipynb` — tract lookup

---
## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import traceback

import traceback


import os

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from astropy.coordinates import SkyCoord
from astropy.table import Table
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import Angle

from lsst.daf.butler import Butler
from lsst.geom import SpherePoint, degrees

print(f"Matplotlib : {matplotlib.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("Imports OK")

In [ ]:
#import ipywidgets as widgets
#%matplotlib widget

In [ ]:
#output dirs
NB_TAG   = 'DP2_DDF_DIAOBJ_BUTLER_01'
DIR_DATA = f'data_{NB_TAG}'
DIR_FIGS = f'figs_{NB_TAG}'
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f'Data : {os.path.abspath(DIR_DATA)}')
print(f'Figs : {os.path.abspath(DIR_FIGS)}')


In [ ]:
# debug flags
DEBUG = False
DEBUG_VISITS = False

## Usefull functions

- from https://github.com/sylvielsstfr/mySIT-COM2025/blob/main/notebooks/2025-05-30-Visits/05_SourcesFromVisits.ipynb

In [ ]:
def getLostOfVisits(registry,where_clause_date):
    """
    Get exposure/visits from butler registry
    parameters:
    ==========
        registry : butler registry
        where_clause_date : specify which instrument and exposure dates in the butler registry
    """
    df_exposure = pd.DataFrame(
        {
        "id": pd.Series(dtype="int"),
        "obs_id": pd.Series(dtype="int"),
        "day_obs": pd.Series(dtype="int"),
        "seq_num": pd.Series(dtype="int"),
        "time_start": pd.Series(dtype="str"),  # ou 'datetime64[ns]' si c’est un datetime
        "time_end": pd.Series(dtype="str"),  # idem
        "type": pd.Series(dtype="str"),
        "target": pd.Series(dtype="str"),
        "filter": pd.Series(dtype="str"),
        "zenith_angle": pd.Series(dtype="float"),
        "expos": pd.Series(dtype="float"),  # ou 'int' selon le cas
        "ra": pd.Series(dtype="float"),
        "dec": pd.Series(dtype="float"),
        "skyangle": pd.Series(dtype="float"),
        "azimuth": pd.Series(dtype="float"),
        "zenith": pd.Series(dtype="float"),
        "science_program": pd.Series(dtype="str"),
        "jd": pd.Series(dtype="float"),
        "mjd": pd.Series(dtype="float"),
        }
    )

    # save the data array in rows before saving in pandas dataframe
    rows = []
    for count, info in enumerate(registry.queryDimensionRecords("exposure", where=where_clause_date)):
        try:
            jd_start = info.timespan.begin.value
            jd_end = info.timespan.end.value
            the_Time_start = Time(jd_start, format="jd", scale="utc")
            the_Time_end = Time(jd_end, format="jd", scale="utc")
            mjd_start = the_Time_start.mjd
            mjd_end = the_Time_end.mjd
            isot_start = the_Time_start.isot
            isot_end = the_Time_end.isot

            if count == 0:
                print("===== Time Conversion Debug Info =====")
                print(f"JD start      : {jd_start} (type: {type(jd_start)})")
                print(f"JD end        : {jd_end} (type: {type(jd_end)})")
                print(f"MJD start     : {mjd_start} (type: {type(mjd_start)})")
                print(f"MJD end       : {mjd_end} (type: {type(mjd_end)})")
                print(f"ISOT start    : {isot_start} (type: {type(isot_start)})")
                print(f"ISOT end      : {isot_end} (type: {type(isot_end)})")
                print("=======================================")

            # put row in a dictionnary before stacking
            row = {
                "id": info.id,
                "obs_id": info.obs_id,
                "day_obs": info.day_obs,
                "seq_num": info.seq_num,
                "time_start": isot_start,
                "time_end": isot_end,
                "type": info.observation_type,
                "target": info.target_name,
                "filter": info.physical_filter,
                "zenith_angle": info.zenith_angle,
                "expos": info.exposure_time,  # Exemple : adapter selon ton objet
                "ra": info.tracking_ra,
                "dec": info.tracking_dec,
                "skyangle": info.sky_angle,
                "azimuth": info.azimuth,
                "zenith": info.zenith_angle,
                "science_program": info.science_program,
                "jd": float(jd_start),
                "mjd": float(mjd_start),
            }
            rows.append(row)

        except ValueError as e:
            print(f"Erreur de valeur : {e}")
        except FileNotFoundError as e:
            print(f"Fichier introuvable : {e}")
        except Exception as e:
            print(f"Erreur inattendue : {type(e).__name__} - {e}")
            print(f">>>   Unexpected error at row {count}:", sys.exc_info()[0])
            traceback.print_exc()  # affiche la stack trace complète


    # Création finale du DataFrame
    df_exposure = pd.DataFrame(rows)
    df_science = df_exposure[df_exposure.type == "science"]
    df_science.reset_index(drop=True, inplace=True)


    return df_science
        


In [ ]:
def FetchTimesForVisits(visit_list):
    """
    Keep for example and possible later use
    """
    # On interroge la table visitDefinition
    Nvisit = len(visit_list)
        
    if Nvisit == 1:
        thevisit = visit_list.values[0]
        rows = registry.queryDimensionRecords("visit", where=f"visit={thevisit}")
    else:
        rows = registry.queryDimensionRecords("visit", where=f"visit in {tuple(visit_list)}")

    # 4. Construire un tableau des résultats
    results = []
    for row in rows:
        visit_id = row.id
        visit_airmass = 1./np.cos(Angle(row.zenith_angle,u.degree).rad)
        visit_azimuth = row.azimuth

        # Extraire l'instant de début de l'observation (Time astropy)
        start_time = row.timespan.begin

        # Convertir en MJD et ISO
        mjd = start_time.to_value("mjd")  # Ex: 60384.28718
        isot = start_time.to_value("isot")  # Ex: '2024-04-19 06:53:32.000'
    
        #mjd = row.startDate.toMjd()
        #utc = Time(mjd, format='mjd', scale='utc').to_value('iso')
        #results.append({"visit": visit_id, "mjd": mjd, "isot": isot})
        results.append({"visit": visit_id, "mjd": mjd, "isot": isot,"airmass":visit_airmass,"azimuth":visit_azimuth})

    df_times = pd.DataFrame(results).sort_values("visit")
    df_times.set_index("visit",inplace=True)
    return df_times


In [ ]:
def RetrieveDRPSources_forTarget(butler,center_coord,datasettype,where_clause,radius_cut=50):
    """
    Keep for example and possible later use
    Find the closest Sourcesthe target_coord 

    parameters:
    - butler
    - the coordinate of the target (center of the cone seach)
    - the datasettype name for the DRP object
    - where_clause : which contrain requirements on the tract and patch numbers
    - cut on angluar separation for the returned for the returned object

    Return
    - object Id with minimum separation , 
    - minimum separation (arcec),
    - the table of DRP objects within the radius_cut
    """

    ra_columns = ['u_ra', 'g_ra', 'r_ra', 'i_ra', 'z_ra', 'y_ra']
    dec_columns = ['u_dec', 'g_dec', 'r_dec', 'i_dec', 'z_dec', 'y_dec']

    
    therefs = butler.registry.queryDatasets(datasettype,  collections=collection, where=where_clause)

    for count,ref in enumerate(therefs):
        the_id = ref.dataId
        the_tract_id = the_id["tract"] 
        print(the_id)
        
        # catalog of rubin objects (a pandas Dataframe) inside the tract
        catalog = butler.get(ref)
        catalog = catalog[catalog["patch"] == patchNbSel] 
       
    
        nobjects = len(catalog)


        # Calcul de la moyenne ligne par ligne, en ignorant les NaN
        catalog['ra'] = catalog[ra_columns].mean(axis=1, skipna=True)
        catalog['dec'] = catalog[dec_columns].mean(axis=1, skipna=True)


        # extract the (ra,dec) coordinates for all te objects of the rubin-catalog
        ra_cat = catalog["ra"].values
        dec_cat = catalog["dec"].values
        # coordinates for all rubin-catalog points
        catalog_coords = SkyCoord(ra=ra_cat*u.deg, dec=dec_cat*u.deg)

        # Angular distance to target
        distances_arcsec = center_coord.separation(catalog_coords).arcsecond

        # add the separation angle to the ctalog
        catalog["sep"] = distances_arcsec


        # closest object from the target
        sepMin = distances_arcsec.min() 
        sepMin_idx = np.where(distances_arcsec == sepMin)[0][0]
    
        closest_obj = catalog[catalog["sep"] <=  sepMin]
                   
        # select a few of these sources to debug the closest candidate
        nearby_obj = catalog[distances_arcsec < radius_cut]
        
        return closest_obj, sepMin, nearby_obj

---
## 1. User Configuration

In [ ]:
SELECTED_DDF    = "COSMOS"   # COSMOS | ECDFS | ELAIS-S1 | XMM-LSS | EDFS-a | EDFS-b | EDFS | M49
CONE_RADIUS_DEG = 1.8
MIN_NSOURCES    = 100

REPO        = "dp2_prep"
COLLECTION  = "LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545"
SKYMAP_NAME = "lsst_cells_v2"
INSTRUMENT = "LSSTCam"
WHERE_CLAUSE_INSTRUMENT = "instrument = '" + f"{INSTRUMENT}" + "'"
DATE_START = 20250415
WHERE_CLAUSE_DATE = WHERE_CLAUSE_INSTRUMENT + f" and day_obs >= {DATE_START}"


TRACT_SEARCH_RADIUS_DEG = 1.8

DDF_COORDS = {
    "COSMOS"   : (150.119, +2.206),
    "ECDFS"    : ( 53.125, -28.100),
    "ELAIS-S1" : (  9.450, -44.000),
    "XMM-LSS"  : ( 35.708,  -4.750),
    "EDFS-a"   : ( 58.900, -49.315),
    "EDFS-b"   : ( 63.600, -47.600),
    "EDFS"     : ( 61.240, -48.423),
    "M49"      : (187.400,  +8.000),
}

RA_CENTER, DEC_CENTER = DDF_COORDS[SELECTED_DDF]
print(f"DDF         : {SELECTED_DDF}  ({RA_CENTER:.4f}°, {DEC_CENTER:+.4f}°)")
print(f"Cone radius : {CONE_RADIUS_DEG}°")
print(f"Min nDiaSrc : {MIN_NSOURCES}")
print(f"Collection  : {COLLECTION}")
print(f"Instrument  : {INSTRUMENT}")
print(f"Where clause  : {WHERE_CLAUSE_INSTRUMENT}")
print(f"Where clause date : {WHERE_CLAUSE_DATE}")

---
## 2. Butler and SkyMap

In [ ]:
butler   = Butler(REPO, collections=COLLECTION)
registry = butler.registry
print("Butler OK")

In [ ]:
try:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME, collections=COLLECTION)
except Exception:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME)
print(f"SkyMap '{SKYMAP_NAME}' : {len(skymap)} tracts")

### 2.1 Search for visit tables

In [ ]:
visitTable_pattern1 = '*isitTable*'
visitTable_pattern2 = '*isitTable'
visitTable_name = "preVisitTable"

In [ ]:
if DEBUG_VISITS:
    dataset_types = registry.queryDatasetTypes(visitTable_pattern1 )
    for dt in dataset_types:
        print(dt.name)

In [ ]:
if DEBUG_VISITS:
    dataset_types = list(butler.registry.queryDatasetTypes(visitTable_pattern1))

    # Pour un affichage lisible (nom du type et storage class)
    for dt in sorted(dataset_types, key=lambda x: x.name):
        print(f"{dt.name:20} | {dt.storageClass.name}")

In [ ]:
if DEBUG_VISITS:
    dataset_types = list(butler.registry.queryDatasetTypes(visitTable_pattern2))
    for dt in dataset_types:
        print(f"{dt.name} :::", dt)
        print(f"    required dimensions: {dt.dimensions.required}")
        print()

## 2.2 Retrieve the science visits in order to match the visitid with the MJD in forced source photometry
- `visitid = id`
- `MJD = mjd `

In [ ]:
visitTable = getLostOfVisits(registry,where_clause_date = WHERE_CLAUSE_DATE)

In [ ]:
print(visitTable.head(1))

---
## 3. Dataset Type Inventory

Identify the correct names and **dimensions** for DIA products — in particular
`dia_object_forced_source` has `(skymap, tract, patch)` dimensions, which
requires iterating over patch refs.

In [ ]:
# Hard-coded after discovery: confirmed present in DM-54249
DIA_OBJ_TYPE    = 'dia_object'               # dims: skymap, tract
DIA_SRC_TYPE    = 'dia_source'               # dims: skymap, tract
FORCED_DIA_TYPE = 'dia_object_forced_source' # dims: skymap, tract, patch

# Print actual dimensions for confirmation
for dt_name in [DIA_OBJ_TYPE, DIA_SRC_TYPE, FORCED_DIA_TYPE]:
    try:
        dt     = registry.getDatasetType(dt_name)
        in_col = registry.queryDatasets(dt_name, collections=COLLECTION).any(
            execute=False, exact=False)
        print(f"{dt_name:40s}  dims={list(dt.dimensions.names)}  present={in_col}")
    except Exception as exc:
        print(f"{dt_name:40s}  ERROR: {exc}")

---
## 4. Find Tracts Covering the DDF

In [ ]:
def find_tracts_for_coord(skymap, ra_deg, dec_deg, radius_deg=1.8):
    cos_dec   = max(np.cos(np.deg2rad(dec_deg)), 0.01)
    step      = 0.35
    found_ids = set()
    for ddec in np.arange(-radius_deg, radius_deg + step, step):
        for dra in np.arange(-radius_deg, radius_deg + step, step):
            if np.sqrt(dra**2 + ddec**2) > radius_deg:
                continue
            ra_s  = ra_deg  + dra / cos_dec
            dec_s = dec_deg + ddec
            if not (-89.9 <= dec_s <= 89.9):
                continue
            sp = SpherePoint(ra_s * degrees, dec_s * degrees)
            try:
                found_ids.add(skymap.findTract(sp).tract_id)
            except Exception:
                pass
    return sorted(found_ids)

tract_ids = find_tracts_for_coord(
    skymap, RA_CENTER, DEC_CENTER, radius_deg=TRACT_SEARCH_RADIUS_DEG
)
print(f"Tracts covering {SELECTED_DDF}: {tract_ids}")

---
## 5. Schema Discovery (one-time probe)

In [ ]:
refs_probe = list(registry.queryDatasets(
    DIA_OBJ_TYPE,
    collections=COLLECTION,
    dataId={"skymap": SKYMAP_NAME, "tract": tract_ids[0]},
))
assert refs_probe, "No dia_object ref found."

obj_raw  = butler.get(refs_probe[0])
df_probe = obj_raw.to_pandas() if hasattr(obj_raw, 'to_pandas') else obj_raw

print(f"index.name  : {df_probe.index.name!r}")
print(f"index.dtype : {df_probe.index.dtype}")
print(f"index[:3]   : {df_probe.index[:3].tolist()}")
print(f"columns     : {list(df_probe.columns)}")

In [ ]:
# Determine ID column
if df_probe.index.name is not None:
    OBJ_ID_COL  = df_probe.index.name
    ID_IN_INDEX = True
    print(f"Object ID in index: '{OBJ_ID_COL}'")
elif any(c in df_probe.columns for c in ['diaObjectId', 'dia_object_id']):
    OBJ_ID_COL  = next(c for c in ['diaObjectId', 'dia_object_id'] if c in df_probe.columns)
    ID_IN_INDEX = False
    print(f"Object ID as column: '{OBJ_ID_COL}'")
else:
    OBJ_ID_COL  = 'row_id'
    ID_IN_INDEX = False
    print("WARNING: no ID found, using row index.")

In [ ]:
# Also probe dia_object_forced_source schema (patch-level)
refs_forced_probe = list(registry.queryDatasets(
    FORCED_DIA_TYPE,
    collections=COLLECTION,
    dataId={"skymap": SKYMAP_NAME, "tract": tract_ids[0]},
))
print(f"dia_object_forced_source refs for tract {tract_ids[0]}: {len(refs_forced_probe)}")

if refs_forced_probe:
    frc_raw   = butler.get(refs_forced_probe[0])
    df_fprobe = frc_raw.to_pandas() if hasattr(frc_raw, 'to_pandas') else frc_raw
    print(f"index.name  : {df_fprobe.index.name!r}")
    print(f"columns     : {list(df_fprobe.columns)}")
    print(f"shape       : {df_fprobe.shape}")

---
## 6. Helper: `to_dataframe()`

In [ ]:
def to_dataframe(obj, id_col=None, id_in_index=None):
    """
    Convert Butler catalog to pandas DataFrame.
    If id_in_index is True, promotes the named index to a regular column.
    Falls back to global OBJ_ID_COL / ID_IN_INDEX if not provided.
    """
    _id_col  = id_col       if id_col       is not None else OBJ_ID_COL
    _id_idx  = id_in_index  if id_in_index  is not None else ID_IN_INDEX

    if isinstance(obj, pd.DataFrame):
        df = obj
    elif hasattr(obj, 'to_pandas'):
        df = obj.to_pandas()
    elif hasattr(obj, 'to_table'):
        df = obj.to_table().to_pandas()
    elif isinstance(obj, Table):
        df = obj.to_pandas()
    else:
        raise TypeError(f"Cannot convert {type(obj)} to DataFrame.")

    if _id_idx and _id_col not in df.columns:
        df = df.copy()
        df.insert(0, _id_col, df.index)
        df = df.reset_index(drop=True)
    elif _id_col == 'row_id' and _id_col not in df.columns:
        df.insert(0, _id_col, range(len(df)))

    return df

---
## 7. Load DiaObject Catalog

In [ ]:
dia_obj_frames = []

for tract_id in tract_ids:
    refs = list(registry.queryDatasets(
        DIA_OBJ_TYPE, collections=COLLECTION,
        dataId={"skymap": SKYMAP_NAME, "tract": tract_id}))
    for ref in refs:
        try:
            df_tmp = to_dataframe(butler.get(ref))
            df_tmp['_tract'] = tract_id
            dia_obj_frames.append(df_tmp)
            print(f"  Tract {tract_id:6d} — {len(df_tmp):,} DiaObjects")
        except Exception as exc:
            print(f"  Tract {tract_id:6d} — ERROR: {exc}")

assert dia_obj_frames, "No dia_object data loaded."
df_dia_obj_all = pd.concat(dia_obj_frames, ignore_index=True)
n_before       = len(df_dia_obj_all)
df_dia_obj_all = df_dia_obj_all.drop_duplicates(subset=OBJ_ID_COL)
print(f"\nTotal: {n_before:,} → {len(df_dia_obj_all):,} after dedup")

In [ ]:
# Detect column names
RA_COL, DEC_COL = next(
    ((ra, dec) for ra, dec in [('ra','dec'), ('coord_ra','coord_dec'), ('RA','DEC')]
     if ra in df_dia_obj_all.columns),
    (None, None)
)
assert RA_COL, f"No RA/Dec found. Columns: {list(df_dia_obj_all.columns)}"

NSRC_COL = next((c for c in ['nDiaSources','n_dia_sources'] if c in df_dia_obj_all.columns), None)

target_order = ['u','g','r','i','z','y']
BANDS = sorted(
    {c.split('_')[0] for c in df_dia_obj_all.columns
     if c.endswith('_psfFluxNdata') and len(c.split('_')[0]) == 1},
    key=lambda b: target_order.index(b) if b in target_order else 99
)
BAND_COLORS = {'u':'purple','g':'green','r':'red','i':'darkorange','z':'sienna','y':'black'}

print(f"ID   : '{OBJ_ID_COL}'")
print(f"RA   : '{RA_COL}'  Dec: '{DEC_COL}'")
print(f"nSrc : '{NSRC_COL}'")
print(f"Bands: {BANDS}")

In [ ]:
# Spatial cone cut
center_sky = SkyCoord(ra=RA_CENTER * u.deg, dec=DEC_CENTER * u.deg)
sep_deg    = center_sky.separation(
    SkyCoord(ra=df_dia_obj_all[RA_COL].values  * u.deg,
             dec=df_dia_obj_all[DEC_COL].values * u.deg)
).deg

df_dia_obj_cone            = df_dia_obj_all[sep_deg <= CONE_RADIUS_DEG].copy()
df_dia_obj_cone['_sep_deg'] = sep_deg[sep_deg <= CONE_RADIUS_DEG]
print(f"In cone: {len(df_dia_obj_cone):,} / {len(df_dia_obj_all):,}")

In [ ]:
# Filter on nDiaSources
if NSRC_COL:
    df_dia_obj_rich = (
        df_dia_obj_cone[df_dia_obj_cone[NSRC_COL] >= MIN_NSOURCES]
        .sort_values(NSRC_COL, ascending=False)
        .reset_index(drop=True)
    )
    print(f"Filtered: {len(df_dia_obj_rich):,}  "
          f"(max={df_dia_obj_rich[NSRC_COL].max()}, "
          f"med={df_dia_obj_rich[NSRC_COL].median():.1f})")
else:
    df_dia_obj_rich = df_dia_obj_cone.copy()
    print("No NSRC_COL — keeping all.")

peek_cols = [c for c in [OBJ_ID_COL, RA_COL, DEC_COL, NSRC_COL, '_tract', '_sep_deg']
             if c in df_dia_obj_rich.columns]
display(df_dia_obj_rich[peek_cols].head(10))

---
## 8. Diagnostic Plots — DiaObject

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
if NSRC_COL:
    ax.hist(df_dia_obj_cone[NSRC_COL].dropna(), bins=100, log=True,
            color='steelblue', alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(MIN_NSOURCES, color='red', ls='--', lw=1.5, label=f'cut ≥ {MIN_NSOURCES}')
ax.set(xlabel='nDiaSources', ylabel='N DiaObjects',
       title=f'{SELECTED_DDF} — All in cone (N={len(df_dia_obj_cone):,})')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1]
if NSRC_COL and len(df_dia_obj_rich) > 0:
    ax.hist(df_dia_obj_rich[NSRC_COL].dropna(), bins=50,
            color='darkorange', alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.set_title(f'{SELECTED_DDF} — Filtered (N={len(df_dia_obj_rich):,})')
ax.set(xlabel='nDiaSources', ylabel='N DiaObjects')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DIR_FIGS}/DiaObj_nSources_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
ax.set_facecolor('#f5f5f5'); ax.invert_xaxis()

ax.scatter(df_dia_obj_cone[RA_COL], df_dia_obj_cone[DEC_COL],
           s=1, color='lightgrey', alpha=0.4, zorder=1,
           label=f'All in cone ({len(df_dia_obj_cone):,})')

if NSRC_COL and len(df_dia_obj_rich) > 0:
    sc = ax.scatter(df_dia_obj_rich[RA_COL], df_dia_obj_rich[DEC_COL],
                    c=df_dia_obj_rich[NSRC_COL], cmap='plasma', s=14, alpha=0.9, zorder=3,
                    norm=mcolors.LogNorm(vmin=df_dia_obj_rich[NSRC_COL].min(),
                                         vmax=df_dia_obj_rich[NSRC_COL].max()),
                    label=f'nSrc ≥ {MIN_NSOURCES} ({len(df_dia_obj_rich):,})')
    plt.colorbar(sc, ax=ax).set_label('nDiaSources (log)', fontsize=10)

ax.plot(RA_CENTER, DEC_CENTER, marker='*', ms=18, color='gold',
        mec='black', mew=1, zorder=10, ls='none', label=f'{SELECTED_DDF} center')
cos_dec = np.cos(np.deg2rad(DEC_CENTER))
theta   = np.linspace(0, 2*np.pi, 361)
ax.plot(RA_CENTER + CONE_RADIUS_DEG/cos_dec*np.cos(theta),
        DEC_CENTER + CONE_RADIUS_DEG*np.sin(theta),
        'r--', lw=1.2, zorder=4, label=f'Cone {CONE_RADIUS_DEG}°')

ax.set(xlabel='RA (deg)', ylabel='Dec (deg)',
       title=f'DP2 DiaObjects — {SELECTED_DDF}  (tracts {tract_ids})')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, ls='--')
plt.tight_layout()
plt.savefig(f'{DIR_FIGS}/DiaObj_sky_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-band flux mean vs sigma
TOP_N  = min(500, len(df_dia_obj_rich))
df_top = df_dia_obj_rich.head(TOP_N)
ncols  = 3
nrows  = int(np.ceil(len(BANDS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows))
axes = np.array(axes).flatten()

for idx, band in enumerate(BANDS):
    ax = axes[idx]
    mc, sc, nc = f'{band}_psfFluxMean', f'{band}_psfFluxSigma', f'{band}_psfFluxNdata'
    if mc not in df_top.columns:
        ax.set_visible(False); continue
    valid = df_top[mc].notna() & (df_top[nc] > 0)
    x = df_top.loc[valid, mc]
    y = df_top.loc[valid, sc] if sc in df_top.columns else np.zeros(valid.sum())
    c = df_top.loc[valid, nc] if nc in df_top.columns else 'grey'
    sc_ = ax.scatter(x, y, c=c, cmap='viridis', s=8, alpha=0.7)
    plt.colorbar(sc_, ax=ax, label='nEpochs')
    ax.set(xlabel=f'{band} psfFluxMean (nJy)', ylabel=f'{band} psfFluxSigma (nJy)',
           title=f'Band {band}  (N={valid.sum():,})')
    ax.title.set_color(BAND_COLORS.get(band, 'black'))
    ax.grid(True, alpha=0.3)

for idx in range(len(BANDS), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle(f'{SELECTED_DDF} — PSF flux summary per band (top {TOP_N} objects)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{DIR_FIGS}/DiaObj_flux_summary_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Epochs per band
fig, ax = plt.subplots(figsize=(10, 5))
for band in BANDS:
    col = f'{band}_psfFluxNdata'
    if col in df_dia_obj_rich.columns:
        vals = df_dia_obj_rich[col].dropna()
        ax.hist(vals, bins=50, alpha=0.55, histtype='stepfilled',
                color=BAND_COLORS.get(band,'grey'), label=f'{band} (med={vals.median():.0f})')
ax.set(xlabel='psfFluxNdata (epochs per band)', ylabel='N DiaObjects',
       title=f'{SELECTED_DDF} — Epochs per band  (N={len(df_dia_obj_rich):,})')
ax.legend(title='Band', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DIR_FIGS}/DiaObj_ndata_band_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Load DiaSources

In [ ]:
rich_ids = set(df_dia_obj_rich[OBJ_ID_COL].values)
print(f"Object IDs to keep: {len(rich_ids):,}")

In [ ]:
dia_src_frames = []
for tract_id in tract_ids:
    refs = list(registry.queryDatasets(
        DIA_SRC_TYPE, collections=COLLECTION,
        dataId={"skymap": SKYMAP_NAME, "tract": tract_id}))
    for ref in refs:
        try:
            df_tmp   = to_dataframe(butler.get(ref))
            join_col = next((c for c in [OBJ_ID_COL, 'diaObjectId', 'dia_object_id']
                             if c in df_tmp.columns), None)
            if join_col:
                df_tmp = df_tmp[df_tmp[join_col].isin(rich_ids)]
            else:
                print(f"  WARNING tract {tract_id}: join col not found. "
                      f"Cols: {list(df_tmp.columns[:10])}")
            dia_src_frames.append(df_tmp)
            print(f"  Tract {tract_id:6d} — {len(df_tmp):,} DiaSources")
        except Exception as exc:
            print(f"  Tract {tract_id:6d} — ERROR: {exc}")

if dia_src_frames:
    df_dia_src_rich = pd.concat(dia_src_frames, ignore_index=True)
    src_id = next((c for c in ['diaSourceId','dia_source_id'] if c in df_dia_src_rich.columns), None)
    if src_id:
        df_dia_src_rich = df_dia_src_rich.drop_duplicates(subset=src_id)
    print(f"\nTotal DiaSources: {len(df_dia_src_rich):,}")
    print(f"Columns: {list(df_dia_src_rich.columns)}")
else:
    print("No DiaSources loaded.")
    df_dia_src_rich = pd.DataFrame()

In [ ]:
if len(df_dia_src_rich) > 0:
    MJD_COL  = next((c for c in ['midPointMjdTai','midpointMjdTai','mjd']
                     if c in df_dia_src_rich.columns), None)
    BAND_COL = next((c for c in ['band','filterName','filter']
                     if c in df_dia_src_rich.columns), None)
    FLUX_COL = next((c for c in ['psfFlux','psf_flux'] if c in df_dia_src_rich.columns), None)
    FERR_COL = next((c for c in ['psfFluxErr','psf_flux_err'] if c in df_dia_src_rich.columns), None)
    print(f"MJD={MJD_COL}  band={BAND_COL}  flux={FLUX_COL}±{FERR_COL}")
else:
    MJD_COL = BAND_COL = FLUX_COL = FERR_COL = None

In [ ]:
if MJD_COL and FLUX_COL and BAND_COL and len(df_dia_src_rich) > 0:
    fig, ax = plt.subplots(figsize=(13, 5))
    for band, grp in df_dia_src_rich.groupby(BAND_COL):
        ax.scatter(grp[MJD_COL], grp[FLUX_COL], s=1, alpha=0.2,
                   color=BAND_COLORS.get(str(band),'grey'), label=str(band))
    ax.set(xlabel='MJD (TAI)', ylabel='PSF flux (nJy)',
           title=f'DiaSources — {SELECTED_DDF} ({len(df_dia_src_rich):,})')
    ax.legend(title='Band', fontsize=9, markerscale=6)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{DIR_FIGS}/DiaSrc_scatter_{SELECTED_DDF}.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("DiaSources scatter skipped.")

---
## 10. Load Forced Photometry (`dia_object_forced_source`)

This dataset has dimensions `(skymap, tract, patch)` — there are **~100 patch refs per tract**.
Each patch file contains per-visit forced-PSF photometry for the DiaObjects in that patch.
We iterate over all patch refs, load, filter on `rich_ids`, and concatenate.

**NOTE :** `dia_object_forced_source` does not contain  nay time, only ('diaObjectId', 'parentObjectId','coord_ra','coord_dec','visit','detector','band',..)
==> this mean one has to fetch the visit table associated to this tract and patch to retreive the MJD  !!! 

In [ ]:
forced_frames = []

# loop on tracts
for tract_id in tract_ids:
    # Query all patch refs for this tract (no patch constraint → all patches)
    refs = list(registry.queryDatasets(
        FORCED_DIA_TYPE,
        collections=COLLECTION,
        dataId={"skymap": SKYMAP_NAME, "tract": tract_id},
    ))
    print(f"Tract {tract_id:6d} — {len(refs)} patch refs")

    # loop on force photometry dia sources refs for that tract 
    for idx,ref in enumerate(refs):
        patch_id = ref.dataId.get('patch', '?')
        #if idx<5:
        #    print(tract_id,patch_id)
        try:
            obj    = butler.get(ref)
            df_tmp = obj.to_pandas() if hasattr(obj, 'to_pandas') else obj

            # Detect join column in this table
            join_col = next(
                (c for c in [OBJ_ID_COL, 'diaObjectId', 'dia_object_id']
                 if c in df_tmp.columns),
                None
            )
            # If ID is in index (same convention as dia_object)
            if join_col is None and df_tmp.index.name in [OBJ_ID_COL, 'diaObjectId', 'dia_object_id']:
                df_tmp = df_tmp.copy()
                df_tmp.insert(0, df_tmp.index.name, df_tmp.index)
                df_tmp = df_tmp.reset_index(drop=True)
                join_col = df_tmp.columns[0]

            if join_col:
                df_filt = df_tmp[df_tmp[join_col].isin(rich_ids)]
            else:
                print(f"    Patch {patch_id}: no join col. "
                      f"idx={df_tmp.index.name!r}  cols={list(df_tmp.columns[:8])}")
                df_filt = df_tmp  # keep all — will be filtered later

            if len(df_filt) > 0:
                df_filt = df_filt.copy()
                df_filt['_tract'] = tract_id
                df_filt['_patch'] = patch_id
                forced_frames.append(df_filt)

        except Exception as exc:
            print(f"    Patch {patch_id} ERROR: {exc}")

if forced_frames:
    df_forced_rich = pd.concat(forced_frames, ignore_index=True)
    print(f"\nTotal forced rows: {len(df_forced_rich):,}")
    print(f"All columns ({len(df_forced_rich.columns)}):")
    print(list(df_forced_rich.columns))
else:
    print("No forced photometry loaded.")
    df_forced_rich = pd.DataFrame()

In [ ]:
# Check names of columns in force photometry table
if len(df_forced_rich) > 0:
    FMJD_COL  = next((c for c in ['midpointMjdTai','midPointMjdTai','mjd']
                      if c in df_forced_rich.columns), None)
    FBAND_COL = next((c for c in ['band','filterName','filter']
                      if c in df_forced_rich.columns), None)
    FFLUX_COL = next((c for c in ['psfFlux','psf_flux']
                      if c in df_forced_rich.columns), None)
    FFERR_COL = next((c for c in ['psfFluxErr','psf_flux_err']
                      if c in df_forced_rich.columns), None)
    FDIFF_COL = next((c for c in ['psfDiffFlux','psf_diff_flux']
                      if c in df_forced_rich.columns), None)
    FDERR_COL = next((c for c in ['psfDiffFluxErr','psf_diff_flux_err']
                      if c in df_forced_rich.columns), None)
    FOBJ_COL  = next((c for c in [OBJ_ID_COL,'diaObjectId','dia_object_id']
                      if c in df_forced_rich.columns), None)

    print(f"MJD  : {FMJD_COL}")
    print(f"Band : {FBAND_COL}")
    print(f"Flux : {FFLUX_COL} ± {FFERR_COL}")
    print(f"Diff : {FDIFF_COL} ± {FDERR_COL}")
    print(f"ObjID: {FOBJ_COL}")
else:
    FMJD_COL = FBAND_COL = FFLUX_COL = FFERR_COL = None
    FDIFF_COL = FDERR_COL = FOBJ_COL = None

In [ ]:
# =========================================================================
# Figure — Forced LC of the most-detected DiaObject
# =========================================================================
if len(df_forced_rich) > 0 and FMJD_COL and FOBJ_COL and FBAND_COL and FFLUX_COL:
    top_id = df_dia_obj_rich.iloc[0][OBJ_ID_COL]
    top_n  = df_dia_obj_rich.iloc[0][NSRC_COL] if NSRC_COL else 'N/A'
    df_lc  = df_forced_rich[df_forced_rich[FOBJ_COL] == top_id].sort_values(FMJD_COL)

    print(f"Forced LC for object {top_id}: {len(df_lc)} rows")

    if len(df_lc) == 0:
        print("WARNING: object found in rich list but no forced rows match. "
              "Check that FOBJ_COL matches OBJ_ID_COL type/value.")
        print(f"  OBJ_ID_COL={OBJ_ID_COL}  FOBJ_COL={FOBJ_COL}")
        print(f"  top_id type={type(top_id)}  forced col dtype={df_forced_rich[FOBJ_COL].dtype}")
    else:
        nrows_fig = 2 if (FDIFF_COL and FDIFF_COL in df_lc.columns) else 1
        fig, axes = plt.subplots(nrows_fig, 1, figsize=(13, 5*nrows_fig), sharex=True)
        if nrows_fig == 1:
            axes = [axes]

        ax = axes[0]
        for band, grp in df_lc.groupby(FBAND_COL):
            yerr = grp[FFERR_COL].values if FFERR_COL else None
            ax.errorbar(grp[FMJD_COL], grp[FFLUX_COL], yerr=yerr,
                        fmt='o', ms=4, alpha=0.8, capsize=2,
                        color=BAND_COLORS.get(str(band),'grey'), label=str(band))
        ax.axhline(0, color='grey', lw=0.8, ls='--')
        ax.set(ylabel='Science PSF flux (nJy)',
               title=f'Forced LC — {OBJ_ID_COL}={top_id}  nDiaSrc={top_n}  [{SELECTED_DDF}]')
        ax.legend(title='Band', fontsize=9); ax.grid(True, alpha=0.3)

        if nrows_fig == 2:
            ax = axes[1]
            for band, grp in df_lc.groupby(FBAND_COL):
                yerr = grp[FDERR_COL].values if FDERR_COL else None
                ax.errorbar(grp[FMJD_COL], grp[FDIFF_COL], yerr=yerr,
                            fmt='o', ms=4, alpha=0.8, capsize=2,
                            color=BAND_COLORS.get(str(band),'grey'), label=str(band))
            ax.axhline(0, color='grey', lw=0.8, ls='--')
            ax.set(ylabel='Diff PSF flux (nJy)')
            ax.legend(title='Band', fontsize=9); ax.grid(True, alpha=0.3)

        axes[-1].set_xlabel('MJD (TAI)', fontsize=11)
        plt.tight_layout()
        plt.savefig(f'{DIR_FIGS}/ForcedLC_{top_id}_{SELECTED_DDF}.png',
                    dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("Forced LC skipped — check Section 10 output above.")

---
## 11. Coordinate Cross-Match with a Fink Alert

In [ ]:
FINK_RA              = 53.125
FINK_DEC             = -28.100
XMATCH_RADIUS_ARCSEC = 1.0

alert_sky  = SkyCoord(ra=FINK_RA*u.deg, dec=FINK_DEC*u.deg)
sep_arcsec = alert_sky.separation(
    SkyCoord(ra=df_dia_obj_cone[RA_COL].values*u.deg,
             dec=df_dia_obj_cone[DEC_COL].values*u.deg)
).arcsec

df_xmatch = df_dia_obj_cone.copy()
df_xmatch['_sep_arcsec'] = sep_arcsec
df_xmatch_cut = (
    df_xmatch[df_xmatch['_sep_arcsec'] <= XMATCH_RADIUS_ARCSEC]
    .sort_values('_sep_arcsec').reset_index(drop=True)
)

print(f"Candidates within {XMATCH_RADIUS_ARCSEC}\": {len(df_xmatch_cut)}")
if len(df_xmatch_cut) > 0:
    show = [c for c in [OBJ_ID_COL, RA_COL, DEC_COL, '_sep_arcsec', NSRC_COL]
            if c in df_xmatch_cut.columns]
    display(df_xmatch_cut[show].head(10))

In [ ]:
if len(df_xmatch_cut) > 0 and len(df_forced_rich) > 0 and FOBJ_COL and FMJD_COL:
    best_id  = df_xmatch_cut.iloc[0][OBJ_ID_COL]
    best_sep = df_xmatch_cut.iloc[0]['_sep_arcsec']
    df_lc    = df_forced_rich[df_forced_rich[FOBJ_COL] == best_id].sort_values(FMJD_COL)

    nrows_fig = 2 if (FDIFF_COL and FDIFF_COL in df_lc.columns) else 1
    fig, axes = plt.subplots(nrows_fig, 1, figsize=(13, 5*nrows_fig), sharex=True)
    if nrows_fig == 1:
        axes = [axes]

    ax = axes[0]
    for band, grp in df_lc.groupby(FBAND_COL):
        yerr = grp[FFERR_COL].values if FFERR_COL else None
        ax.errorbar(grp[FMJD_COL], grp[FFLUX_COL], yerr=yerr,
                    fmt='o', ms=4, alpha=0.85, capsize=2,
                    color=BAND_COLORS.get(str(band),'grey'), label=str(band))
    ax.axhline(0, color='grey', lw=0.8, ls='--')
    ax.set(ylabel='PSF flux (nJy)',
           title=f'Cross-match LC — {OBJ_ID_COL}={best_id}  sep={best_sep:.3f}"\n'
                 f'Fink: RA={FINK_RA:.5f}°  Dec={FINK_DEC:+.5f}°')
    ax.legend(title='Band', fontsize=9); ax.grid(True, alpha=0.3)

    if nrows_fig == 2:
        ax = axes[1]
        for band, grp in df_lc.groupby(FBAND_COL):
            yerr = grp[FDERR_COL].values if FDERR_COL else None
            ax.errorbar(grp[FMJD_COL], grp[FDIFF_COL], yerr=yerr,
                        fmt='o', ms=4, alpha=0.85, capsize=2,
                        color=BAND_COLORS.get(str(band),'grey'), label=str(band))
        ax.axhline(0, color='grey', lw=0.8, ls='--')
        ax.set(ylabel='Diff PSF flux (nJy)')
        ax.legend(title='Band', fontsize=9); ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('MJD (TAI)', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'{DIR_FIGS}/XmatchLC_{best_id}_{SELECTED_DDF}.png',
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Cross-match LC skipped.")

---
## 12. Save Results

In [ ]:
for df, name in [
    (df_dia_obj_rich, f'DiaObjects_{SELECTED_DDF}_nmin{MIN_NSOURCES}.csv'),
    (df_dia_src_rich, f'DiaSources_{SELECTED_DDF}_nmin{MIN_NSOURCES}.csv'),
    (df_forced_rich,  f'ForcedSrc_{SELECTED_DDF}_nmin{MIN_NSOURCES}.csv'),
]:
    if len(df) > 0:
        path = f'{DIR_DATA}/{name}'
        df.to_csv(path, index=False)
        print(f"Saved: {path}  ({len(df):,} rows)")